
<img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
<h1 align="left"> DSPy Facility Support Analyzer Demo </h1>
Delivered by WeCloudData
<br>


## Learning Objectives

This demo illustrates the following concepts and techniques:

1.  **Programatic Prompt Optimization with GEPA**: Learn how to use GEPA to automatically improve the prompts for language model-based predictors in a program.
2.  **Analyzing Optimized Prompts**: Understand how to inspect the prompts generated by GEPA and interpret the changes it made based on the optimization process.
3.  **Evaluating Model Performance**: See how to evaluate the performance of the program with the optimized prompts and compare it to the performance with the original prompts.
4.  **Applying GEPA to Real-World Tasks**: Gain insight into how GEPA can be applied to improve the performance of language models on various classification and prediction tasks.

### Setup and Data Preparation

First, we'll install the necessary libraries and load the dataset.

#### Install Libraries

We install `dspy` for building and optimizing the language model program, and `mlflow` for tracking experiments (although MLflow tracking isn't explicitly used in this simplified version).

In [ ]:
!pip install dspy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 285.2/285.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 70.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 72.0 MB/s eta 0:00:00
  Attempting uninstall: grpcio
    Found existing installation: grpcio 1.76.0
    Uninstalling grpcio-1.76.0:
      Successfully uninstalled grpcio-1.76.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
grpcio-status 1.71.2 requires grpcio>=1.71.2, but you have grpcio 1.67.1 which is incompatible.


#### Initialize DSPy and Language Model

We initialize DSPy and configure the language model that our program will use. We'll use a placeholder model name here and retrieve the OpenAI API key securely.

In [ ]:
import dspy
from google.colab import userdata

api_key = userdata.get("OPENAI_API_KEY")
lm = dspy.LM("openai/gpt-4.1-nano", temperature=1, api_key=api_key)
dspy.configure(lm=lm)

#### Initialize Dataset

This function downloads the dataset from a public URL, which contains examples of facility support messages and their desired classifications. It then splits the data into training, validation, and testing sets.

In [ ]:
import requests
import dspy
import json
import random

def init_dataset():
    # Load from the url
    url = "https://raw.githubusercontent.com/meta-llama/llama-prompt-ops/refs/heads/main/use-cases/facility-support-analyzer/dataset.json"
    dataset = json.loads(requests.get(url).text)
    dspy_dataset = [
        dspy.Example({
            "message": d['fields']['input'],
            "answer": d['answer'],
        }).with_inputs("message")
        for d in dataset
    ]
    random.Random(0).shuffle(dspy_dataset)
    train_set = dspy_dataset[:int(len(dspy_dataset) * 0.33)]
    val_set = dspy_dataset[int(len(dspy_dataset) * 0.33):int(len(dspy_dataset) * 0.66)]
    test_set = dspy_dataset[int(len(dspy_dataset) * 0.66):]

    return train_set, val_set, test_set

#### Load and Split Data

Here, we execute the `init_dataset` function and see the number of examples in each split.

In [ ]:
train_set, val_set, test_set = init_dataset()

len(train_set), len(val_set), len(test_set)

(66, 66, 68)

**Explanation of Output:**

This output shows the number of examples in each of the three datasets: training (`train_set`), validation (`val_set`), and testing (`test_set`). This confirms that the data was loaded and split correctly.

#### Display Sample Data

Let's look at an example from the training set to understand the input message format and the expected output structure.

In [ ]:
print("Input Message:")
print(train_set[0]['message'])

print("\n\nGold Answer:")
for k, v in json.loads(train_set[0]['answer']).items():
    print(f"{k}: {v}")

Input Message:
Subject: Adjusting Bi-Weekly Cleaning Schedule for My Office

Dear ProCare Facility Solutions Support Team,

I hope this message finds you well. My name is Dr. Alex Turner, and I have been utilizing your services for my office space for the past year. I must say, your team's dedication to maintaining a pristine environment has been commendable and greatly appreciated.

I am reaching out to discuss the scheduling of our regular cleaning services. While I find the logistical challenges of coordinating these services intellectually stimulating, I believe we could optimize the current schedule to better suit the needs of my team and our workflow. Specifically, I would like to explore the possibility of adjusting our cleaning schedule to a bi-weekly arrangement, ideally on Tuesdays and Fridays, to ensure our workspace remains consistently clean without disrupting our research activities.

Previously, I have attempted to adjust the schedule through the online portal, but I enc

**Explanation of Output:**

This output displays the content of the first example in the `train_set`.

*   **Input Message:** This is the raw text of a facility support request email.
*   **Gold Answer:** This is the expected classification for the input message, provided as a JSON object. It includes the correct categories, sentiment (`neutral`), and urgency (`low`) for this specific message.

### Building the DSPy Program

We define the structure of our program using DSPy Signatures and Modules.

#### Define Signatures and Module

We define signatures for the three tasks: determining urgency, sentiment, and categories. A `FacilitySupportAnalyzerMM` module is created to combine these tasks using `ChainOfThought` to encourage step-by-step reasoning by the language model.

In [ ]:
from typing import List, Literal


class FacilitySupportAnalyzerUrgency(dspy.Signature):
    """
    Read the provided message and determine the urgency.
    """
    message: str = dspy.InputField()
    urgency: Literal['low', 'medium', 'high'] = dspy.OutputField()

class FacilitySupportAnalyzerSentiment(dspy.Signature):
    """
    Read the provided message and determine the sentiment.
    """
    message: str = dspy.InputField()
    sentiment: Literal['positive', 'neutral', 'negative'] = dspy.OutputField()

class FacilitySupportAnalyzerCategories(dspy.Signature):
    """
    Read the provided message and determine the set of categories applicable to the message.
    """
    message: str = dspy.InputField()
    categories: List[Literal["emergency_repair_services", "routine_maintenance_requests", "quality_and_safety_concerns", "specialized_cleaning_services", "general_inquiries", "sustainability_and_environmental_practices", "training_and_support_requests", "cleaning_services_scheduling", "customer_feedback_and_complaints", "facility_management_issues"]] = dspy.OutputField()

class FacilitySupportAnalyzerMM(dspy.Module):
    def __init__(self):
        self.urgency_module = dspy.ChainOfThought(FacilitySupportAnalyzerUrgency)
        self.sentiment_module = dspy.ChainOfThought(FacilitySupportAnalyzerSentiment)
        self.categories_module = dspy.ChainOfThought(FacilitySupportAnalyzerCategories)

    def forward(self, message: str):
        urgency = self.urgency_module(message=message)
        sentiment = self.sentiment_module(message=message)
        categories = self.categories_module(message=message)

        return dspy.Prediction(
            urgency=urgency.urgency,
            sentiment=sentiment.sentiment,
            categories=categories.categories
        )

program = FacilitySupportAnalyzerMM()

### Evaluating the Program

We define metrics to evaluate how well our program performs and then run an initial evaluation on the test set.

#### Define Scoring Metrics

These functions compare the program's predictions with the gold standard labels for urgency, sentiment, and categories. The main `metric` function computes an overall score.

In [ ]:
def score_urgency(gold_urgency, pred_urgency):
    """
    Compute score for the urgency module.
    """
    score = 1.0 if gold_urgency == pred_urgency else 0.0
    return score

def score_sentiment(gold_sentiment, pred_sentiment):
    """
    Compute score for the sentiment module.
    """
    score = 1.0 if gold_sentiment == pred_sentiment else 0.0
    return score

def score_categories(gold_categories, pred_categories):
    """
    Compute score for the categories module.
    Uses the same match/mismatch logic as category accuracy in the score.
    """
    correct = 0
    for k, v in gold_categories.items():
        if v and k in pred_categories:
            correct += 1
        elif not v and k not in pred_categories:
            correct += 1
    score = correct / len(gold_categories)
    return score

def metric(example, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Computes a score based on agreement between prediction and gold standard for categories, sentiment, and urgency.
    Returns the score (float).
    """
    # Parse gold standard from example
    gold = json.loads(example['answer'])

    # Compute scores for all modules
    score_urgency_val = score_urgency(gold['urgency'], pred.urgency)
    score_sentiment_val = score_sentiment(gold['sentiment'], pred.sentiment)
    score_categories_val = score_categories(gold['categories'], pred.categories)

    # Overall score: average of the three accuracies
    total = (score_urgency_val + score_sentiment_val + score_categories_val) / 3

    return total

In [ ]:
import dspy
evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    num_threads=32,
    display_table=True,
    display_progress=True
)

evaluate(program)

Average Metric: 51.07 / 68 (75.1%): 100%|██████████| 68/68 [00:13<00:00,  5.17it/s]

2025/12/18 15:34:18 INFO dspy.evaluate.evaluate: Average Metric: 51.06666666666666 / 68 (75.1%)


,message,answer,urgency,sentiment,categories,metric
0,"Hey ProCare Support Team, Hope you all are doing great! My name is...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,[sustainability_and_environmental_practices],✔️ [1.000]
1,"Hey ProCare Team, Hope you’re all doing well! My name’s Jake, and ...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,positive,[routine_maintenance_requests],✔️ [1.000]
2,"Subject: Assistance Needed for HVAC Maintenance Hi [Receiver], I h...","{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",medium,neutral,[routine_maintenance_requests],✔️ [1.000]
3,Subject: A Green Inquiry from a Bill Maher Enthusiast Hey ProCare ...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,"[sustainability_and_environmental_practices, general_inquiries]",✔️ [0.633]
4,Subject: Inquiry on Sustainability Practices Dear ProCare Facility...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,neutral,[sustainability_and_environmental_practices],✔️ [0.667]
...,...,...,...,...,...,...
63,Subject: Inquiry About Your Eco-Friendly Practices Dear ProCare Fa...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",low,positive,[sustainability_and_environmental_practices],✔️ [0.600]
64,Subject: Assistance Needed for Facility Management Issue Dear ProC...,"{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,[facility_management_issues],✔️ [1.000]
65,"Subject: Request for Training and Support Hi ProCare Support Team,...","{""categories"": {""routine_maintenance_requests"": false, ""customer_f...",medium,positive,[training_and_support_requests],✔️ [0.667]
66,Subject: Concerns About Studio Maintenance and Rent Increase Dear ...,"{""categories"": {""routine_maintenance_requests"": true, ""customer_fe...",high,negative,"[routine_maintenance_requests, quality_and_safety_concerns]",✔️ [0.267]


EvaluationResult(score=75.1, results=<list of 68 results>)

**Explanation of Output:**

This output shows the results of the initial evaluation of the `program` (the unoptimized `FacilitySupportAnalyzerMM` module) on the `test_set`.

*   **Average Metric:** This indicates the overall performance of the program on the test set, calculated as the average of the scores for urgency, sentiment, and categories across all examples. A score of 75% means that, on average, the program correctly classified about 75% of the aspects (urgency, sentiment, and individual categories) for each message in the test set.
*   **Display Table:** This table provides a detailed breakdown of the evaluation results for each example in the test set.
    *   `message`: The input message.
    *   `answer`: The gold standard classification (the correct answer).
    *   `urgency`, `sentiment`, `categories`: The predictions made by the unoptimized program.
    *   `metric`: The overall metric score for that specific example. `✔️` indicates a perfect score (1.0) for that example.

This evaluation provides a baseline performance for the program before any optimization.

### Optimizing the Program with GEPA

Now, we use the GEPA optimizer to automatically improve our program's performance by generating better prompts or adjusting the program structure based on feedback.

#### Define Metric with Feedback

We define a modified metric that also provides feedback to the optimizer about which specific predictions were incorrect.

In [ ]:
import json
import dspy

def feedback_urgency(gold_urgency, pred_urgency):
    """
    Generate feedback for the urgency module.
    """
    score = 1.0 if gold_urgency == pred_urgency else 0.0
    if gold_urgency == pred_urgency:
        feedback = f"You correctly classified the urgency of the message as `{gold_urgency}`. This message is indeed of `{gold_urgency}` urgency."
    else:
        feedback = f"You incorrectly classified the urgency of the message as `{pred_urgency}`. The correct urgency is `{gold_urgency}`. Think about how you could have reasoned to get the correct urgency label."
    return feedback, score

def feedback_sentiment(gold_sentiment, pred_sentiment):
    """
    Generate feedback for the sentiment module.
    """
    score = 1.0 if gold_sentiment == pred_sentiment else 0.0
    if gold_sentiment == pred_sentiment:
        feedback = f"You correctly classified the sentiment of the message as `{gold_sentiment}`. This message is indeed `{gold_sentiment}`."
    else:
        feedback = f"You incorrectly classified the sentiment of the message as `{pred_sentiment}`. The correct sentiment is `{gold_sentiment}`. Think about how you could have reasoned to get the correct sentiment label."
    return feedback, score

def feedback_categories(gold_categories, pred_categories):
    """
    Generate feedback for the categories module.
    Uses the same match/mismatch logic as category accuracy in the score.
    """
    correctly_included = [k for k, v in gold_categories.items() if v and k in pred_categories]
    incorrectly_included = [k for k, v in gold_categories.items() if not v and k in pred_categories]
    incorrectly_excluded = [k for k, v in gold_categories.items() if v and k not in pred_categories]
    correctly_excluded = [k for k, v in gold_categories.items() if not v and k not in pred_categories]  # For completeness in accuracy check

    # Recompute category accuracy (matches score logic)
    score = (len(correctly_included) + len(correctly_excluded)) / len(gold_categories)

    if score == 1.0:
        fb_text = f"The category classification is perfect. You correctly identified that the message falls under the following categories: `{repr(correctly_included)}`."
    else:
        fb_text = f"The category classification is not perfect. You correctly identified that the message falls under the following categories: `{repr(correctly_included)}`.\n"
        if incorrectly_included:
            fb_text += f"However, you incorrectly identified that the message falls under the following categories: `{repr(incorrectly_included)}`. The message DOES NOT fall under these categories.\n"
        if incorrectly_excluded:
            prefix = "Additionally, " if incorrectly_included else "However, "
            fb_text += f"{prefix}you didn't identify the following categories that the message actually falls under: `{repr(incorrectly_excluded)}`.\n"
        fb_text += "Think about how you could have reasoned to get the correct category labels."
    return fb_text, score

def metric_with_feedback(example, pred, trace=None, pred_name=None, pred_trace=None):
    """
    Computes a score based on agreement between prediction and gold standard for categories, sentiment, and urgency.
    Optionally provides feedback text for a specific predictor module, using the same comparison logic as the score.
    Returns a dspy.Prediction with score (float) and feedback (str).
    """
    # Parse gold standard from example
    gold = json.loads(example['answer'])

    # Compute feedback and scores for all modules
    fb_urgency, score_urgency = feedback_urgency(gold['urgency'], pred.urgency)
    fb_sentiment, score_sentiment = feedback_sentiment(gold['sentiment'], pred.sentiment)
    fb_categories, score_categories = feedback_categories(gold['categories'], pred.categories)

    # Overall score: average of the three accuracies
    total = (score_urgency + score_sentiment + score_categories) / 3

    if pred_name is None:
        return total

    elif pred_name == 'urgency_module.predict':
        feedback = fb_urgency
    elif pred_name == 'sentiment_module.predict':
        feedback = fb_sentiment
    elif pred_name == 'categories_module.predict':
        feedback = fb_categories

    return dspy.Prediction(score=total, feedback=feedback)

#### Initialize GEPA Optimizer

We initialize the GEPA optimizer, specifying the feedback metric and the reflection model it should use.

In [ ]:
from dspy import GEPA

optimizer = GEPA(
    metric=metric_with_feedback,
    auto="light", # <-- We will use a light budget for this tutorial. However, we typically recommend using auto="heavy" for optimized performance!
    num_threads=32,
    track_stats=True,
    use_merge=False,
    reflection_lm=dspy.LM(model="gpt-5", temperature=1.0, max_tokens=32000, api_key=api_key)
)

#### Compile (Optimize) the Program

This is the core optimization step. GEPA will run iterations, generating new versions of the prompts and evaluating them using the feedback metric until the optimization budget is met or performance converges.

In [ ]:
optimized_program = optimizer.compile(
    program,
    trainset=train_set,
    valset=val_set,
)

2025/12/18 15:34:44 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 1643 metric calls of the program. This amounts to 12.45 full evals on the train+val set.
2025/12/18 15:34:44 INFO dspy.teleprompt.gepa.gepa: Using 66 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.
GEPA Optimization:   0%|          | 0/1643 [00:00<?, ?rollouts/s]2025/12/18 15:34:55 INFO dspy.evaluate.evaluate: Average Metric: 48.53333333333333 / 66 (73.5%)
2025/12/18 15:34:55 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.7353535353535353
GEPA Optimization:   4%|▍         | 66/1643 [00:11<04:38,  5.67rollouts/s]2025/12/18 15:34:55 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.7

Average Metric: 2.30 / 3 (76.7%): 100%|██████████| 3/3 [00:03<00:00,  1.23s/it]

2025/12/18 15:34:59 INFO dspy.evaluate.evaluate: Average Metric: 2.3 / 3 (76.7%)


2025/12/18 15:35:58 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Proposed new text for urgency_module.predict: You are given a single input field:
- message: Full text of a customer email sent to ProCare Facility Solutions (a facilities management provider offering office maintenance, cleaning services, and HVAC maintenance).

Your task:
- Read the message and determine its urgency.
- Provide a brief justification and a single-word urgency label.

Domain context and examples to anchor your judgment:
- Typical topics include: cleaning routines (including high-traffic spaces like main lobbies and conference rooms), HVAC system performance/maintenance, facility safety/quality, and general information (e.g., eco-friendly practices).
- Minor safety concerns related to HVAC and inconsistencies in cleaning routines in busy areas warrant timely attention but are not necessarily emergencies.
- Routine HVAC maintenance with a requested visit within the next week or two (e.g., ahead of an event) 

Average Metric: 2.93 / 3 (97.8%): 100%|██████████| 3/3 [00:03<00:00,  1.28s/it]

...

Average Metric: 2.47 / 3 (82.2%): 100%|██████████| 3/3 [00:03<00:00,  1.15s/it]

2025/12/18 16:18:35 INFO dspy.evaluate.evaluate: Average Metric: 2.466666666666667 / 3 (82.2%)


2025/12/18 16:19:16 INFO dspy.teleprompt.gepa.gepa: Iteration 39: Proposed new text for urgency_module.predict: You are given one input field:
- message: the raw text of a customer email or message.

Your task:
- Read the message and classify its urgency.
- Output two fields ONLY:
  - reasoning: 1–3 concise sentences explaining why you chose the urgency, explicitly referencing concrete cues in the message text (e.g., outage, hazard, explicit timing words).
  - urgency: one of "low", "medium", or "high" (lowercase).

Do not provide solutions, troubleshooting steps, apologies, or extra commentary—only the reasoning and the urgency label.

Strict decision rubric (apply in this order; classify by the highest applicable rule):

3) High urgency
- Active failures/outages or conditions that threaten safety, property, or critical operations, OR explicit urgent timing.
- Indicators include any of:
  - Explicit urgency/time demands: “urgent,” “ASAP,” “immediately,” “emergency,” “today,” “by end o

Average Metric: 2.60 / 3 (86.7%): 100%|██████████| 3/3 [00:03<00:00,  1.10s/it]

2025/12/18 16:19:21 INFO dspy.evaluate.evaluate: Average Metric: 2.6 / 3 (86.7%)


#### Display Optimized Prompts

After optimization, we can inspect the prompts that GEPA generated for each predictor in the program.

In [ ]:
for name, pred in optimized_program.named_predictors():
    print("================================")
    print(f"Predictor: {name}")
    print("================================")
    print("Prompt:")
    print(pred.signature.instructions)
    print("*********************************")

**Explanation of Output:**

This output displays the prompts that were used for each of the predictors within the `optimized_program` after the GEPA optimization process.

*   **Predictor:** The name of the predictor module (e.g., `urgency_module.predict`).
*   **Prompt:** The instructions and examples (if any were added by the optimizer) that were used to guide the language model for this specific task.

By comparing these prompts to the original signatures, you can see how GEPA has potentially modified the instructions to improve the model's performance on the task based on the training and validation data and the feedback metric.